# 🐍 Snake Game v2 - 개선 버전

25×25 그리드의 개선된 스네이크 게임입니다.

**v2 개선 사항:**
- 25×25 격자판으로 확장 (셀 크기 22px)
- 둥근 뱀 세그먼트 + 그라디언트 색상
- 게임 오버 시 캔버스 내 오버레이 표시 (알림창 없음)
- 100점마다 레벨 업 (v1의 2배)
- 속도 조정: 200ms 시작, 레벨당 5ms 감소, 최소 100ms
- 네온 초록 뱀, 밝은 먹이 색상
- 점수 / 레벨 / 최고점수(localStorage) HUD
- R키로 재시작

## 📦 CSS 스타일 정의

In [ ]:
css = """
/* 전체 요소 초기화 */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

/* 바디 배경 및 중앙 정렬 */
body {
    background: linear-gradient(135deg, #0d0d1a 0%, #1a1a2e 50%, #0d0d1a 100%);
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 100vh;
    font-family: 'Courier New', Courier, monospace;
    color: #e0e0e0;
}

/* 메인 게임 컨테이너 */
.game-container {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 14px;
    padding: 16px;
}

/* 게임 제목 스타일 */
.game-title {
    font-size: 30px;
    font-weight: bold;
    color: #00ff88;
    letter-spacing: 5px;
    text-transform: uppercase;
    text-shadow:
        0 0 10px rgba(0, 255, 136, 0.8),
        0 0 20px rgba(0, 255, 136, 0.4),
        0 0 40px rgba(0, 255, 136, 0.2);
}

/* 버전 뱃지 */
.version-badge {
    font-size: 11px;
    color: #00ff88;
    background: rgba(0, 255, 136, 0.1);
    border: 1px solid rgba(0, 255, 136, 0.3);
    border-radius: 12px;
    padding: 2px 10px;
    letter-spacing: 2px;
    margin-top: -8px;
}

/* HUD 점수판 컨테이너 */
.hud {
    display: flex;
    gap: 0;
    background: rgba(0, 0, 0, 0.4);
    border: 1px solid rgba(0, 255, 136, 0.25);
    border-radius: 10px;
    overflow: hidden;
}

/* HUD 개별 항목 */
.hud-item {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 2px;
    padding: 8px 24px;
    border-right: 1px solid rgba(0, 255, 136, 0.15);
}

/* 마지막 HUD 항목 구분선 제거 */
.hud-item:last-child {
    border-right: none;
}

/* HUD 레이블 텍스트 */
.hud-label {
    font-size: 9px;
    color: #6b7280;
    letter-spacing: 2.5px;
    text-transform: uppercase;
}

/* HUD 숫자 값 */
.hud-value {
    font-size: 24px;
    font-weight: bold;
    color: #00ff88;
    font-variant-numeric: tabular-nums;
    text-shadow: 0 0 8px rgba(0, 255, 136, 0.5);
}

/* 게임 캔버스 */
canvas {
    border: 2px solid rgba(0, 255, 136, 0.35);
    border-radius: 6px;
    display: block;
    box-shadow:
        0 0 30px rgba(0, 255, 136, 0.1),
        inset 0 0 30px rgba(0, 0, 0, 0.3);
}

/* 조작 안내 텍스트 */
.instructions {
    font-size: 11px;
    color: #4b5563;
    letter-spacing: 1.5px;
    text-align: center;
}
"""

## 🏗️ HTML 구조

In [ ]:
html_body = """
<div class="game-container">
    <!-- 게임 제목 및 버전 -->
    <div class="game-title">🐍 Snake</div>
    <div class="version-badge">v2 · 개선 버전</div>

    <!-- 점수 HUD -->
    <div class="hud">
        <div class="hud-item">
            <span class="hud-label">Score</span>
            <span class="hud-value" id="score">0</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">Level</span>
            <span class="hud-value" id="level">1</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">Best</span>
            <span class="hud-value" id="best">0</span>
        </div>
    </div>

    <!-- 게임 캔버스 (25x25 그리드 = 550px) -->
    <canvas id="gameCanvas" width="550" height="550"></canvas>

    <!-- 조작 안내 -->
    <div class="instructions">방향키로 이동 &nbsp;·&nbsp; R 키로 재시작</div>
</div>
"""

## 🎮 JavaScript 게임 로직

In [ ]:
js = """
// 캔버스와 2D 렌더링 컨텍스트 초기화
const canvas = document.getElementById('gameCanvas');
const ctx = canvas.getContext('2d');

// 게임 상수 설정 (v2: 25x25 그리드)
const GRID_SIZE = 25;          // 그리드 크기 (25x25)
const CELL_SIZE = 22;          // 각 셀의 픽셀 크기
const BASE_SPEED = 200;        // 초기 이동 속도 (밀리초)
const SPEED_DECREASE = 5;      // 레벨당 속도 감소량 (v1보다 완만)
const MIN_SPEED = 100;         // 최소 이동 속도
const POINTS_PER_FOOD = 10;    // 먹이당 획득 점수
const POINTS_PER_LEVEL = 100;  // 레벨 업에 필요한 점수 (v1의 2배)
const CORNER_RADIUS = 4;       // 뱀 세그먼트 모서리 둥글기

// v2 네온 색상 팔레트
const COLOR_BG = '#0d0d1a';          // 배경색 (더 짙은 남색)
const COLOR_GRID = '#111128';        // 그리드 선 (거의 보이지 않게)
const COLOR_SNAKE_HEAD = '#00ff88';  // 뱀 머리 (네온 초록)
const COLOR_SNAKE_MID = '#00cc66';   // 뱀 중간 색상
const COLOR_SNAKE_TAIL = '#008844';  // 뱀 꼬리 색상 (어두운 초록)
const COLOR_FOOD = '#ff6b35';        // 먹이 색상 (주황)
const COLOR_FOOD_INNER = '#ffcc00';  // 먹이 내부 (노란색)
const COLOR_OVERLAY = 'rgba(0, 0, 0, 0.80)'; // 게임오버 오버레이

// 게임 상태 변수 초기화
let snake = [];          // 뱀 세그먼트 배열 [{x, y}, ...]
let food = {};           // 먹이 위치 {x, y}
let direction = {};      // 현재 이동 방향 {x, y}
let nextDirection = {};  // 입력된 다음 방향
let score = 0;           // 현재 게임 점수
let level = 1;           // 현재 레벨
let bestScore = 0;       // localStorage 최고 점수
let gameOver = false;    // 게임 오버 상태 플래그
let gameLoop = null;     // setInterval 반환 ID
let gameStarted = false; // 게임 진행 중 여부
let animFrame = 0;       // 애니메이션 프레임 카운터 (먹이 펄스 효과)

// localStorage에서 최고 점수 불러오기
function loadBestScore() {
    try {
        // 저장된 v2 최고 점수 조회
        const saved = localStorage.getItem('snakeV2Best');
        bestScore = saved ? parseInt(saved, 10) : 0;
    } catch (e) {
        // 접근 불가 시 기본값 0
        bestScore = 0;
    }
    // HUD에 최고 점수 반영
    document.getElementById('best').textContent = bestScore;
}

// 최고 점수를 localStorage에 저장
function saveBestScore() {
    try {
        // 현재 점수가 기존 최고 점수보다 높을 때만 저장
        if (score > bestScore) {
            bestScore = score;
            localStorage.setItem('snakeV2Best', bestScore);
            document.getElementById('best').textContent = bestScore;
        }
    } catch (e) {
        // 저장 실패 시 무시 (비공개 창 등)
    }
}

// 게임 초기화 및 시작
function initGame() {
    // 기존 게임 루프 중단
    if (gameLoop) clearInterval(gameLoop);

    // 뱀 초기 위치: 그리드 중앙, 길이 4
    const center = Math.floor(GRID_SIZE / 2);
    snake = [
        { x: center,     y: center },      // 머리
        { x: center - 1, y: center },      // 몸통 1
        { x: center - 2, y: center },      // 몸통 2
        { x: center - 3, y: center }       // 꼬리
    ];

    // 초기 이동 방향: 오른쪽
    direction = { x: 1, y: 0 };
    nextDirection = { x: 1, y: 0 };

    // 게임 상태 초기화
    score = 0;
    level = 1;
    gameOver = false;
    gameStarted = true;
    animFrame = 0;

    // HUD 초기화
    document.getElementById('score').textContent = 0;
    document.getElementById('level').textContent = 1;

    // 먹이 배치
    spawnFood();

    // 게임 루프 시작
    gameLoop = setInterval(update, BASE_SPEED);
}

// 뱀과 겹치지 않는 위치에 먹이 생성
function spawnFood() {
    let pos;
    let attempts = 0;
    do {
        // 랜덤 좌표 생성
        pos = {
            x: Math.floor(Math.random() * GRID_SIZE),
            y: Math.floor(Math.random() * GRID_SIZE)
        };
        attempts++;
        // 뱀과 겹치지 않을 때까지 반복 (최대 1000번)
    } while (snake.some(seg => seg.x === pos.x && seg.y === pos.y) && attempts < 1000);
    food = pos;
}

// 레벨에 따른 게임 속도 계산
function getSpeed() {
    // 레벨이 높을수록 간격이 줄어듦 (빨라짐)
    const speed = BASE_SPEED - (level - 1) * SPEED_DECREASE;
    return Math.max(speed, MIN_SPEED);
}

// 매 틱마다 호출되는 게임 업데이트 함수
function update() {
    // 입력된 방향으로 업데이트
    direction = { ...nextDirection };
    animFrame++; // 애니메이션 카운터 증가

    // 새 머리 위치 계산
    const head = {
        x: snake[0].x + direction.x,
        y: snake[0].y + direction.y
    };

    // 벽 충돌 감지
    if (head.x < 0 || head.x >= GRID_SIZE || head.y < 0 || head.y >= GRID_SIZE) {
        endGame(); // 경계 이탈 → 게임 오버
        return;
    }

    // 자기 몸 충돌 감지
    if (snake.some(seg => seg.x === head.x && seg.y === head.y)) {
        endGame(); // 자기 몸 충돌 → 게임 오버
        return;
    }

    // 머리를 앞으로 이동
    snake.unshift(head);

    // 먹이와의 충돌 확인
    if (head.x === food.x && head.y === food.y) {
        // 점수 추가
        score += POINTS_PER_FOOD;
        document.getElementById('score').textContent = score;

        // 레벨 업 여부 확인
        const newLevel = Math.floor(score / POINTS_PER_LEVEL) + 1;
        if (newLevel > level) {
            level = newLevel;
            document.getElementById('level').textContent = level;
            // 속도 재조정
            clearInterval(gameLoop);
            gameLoop = setInterval(update, getSpeed());
        }

        // 새 먹이 생성 (꼬리 제거 없음 = 뱀 길이 증가)
        spawnFood();
    } else {
        // 먹이 없으면 꼬리 제거 (이동만)
        snake.pop();
    }

    // 화면 갱신
    draw();
}

// 게임 종료 처리
function endGame() {
    // 게임 루프 정지
    clearInterval(gameLoop);
    gameOver = true;

    // 최고 점수 갱신 및 저장
    saveBestScore();

    // 게임 오버 화면 표시
    draw();
    drawGameOver();
}

// 캔버스에 게임 오버 오버레이 그리기
function drawGameOver() {
    // 어두운 반투명 오버레이 적용
    ctx.fillStyle = COLOR_OVERLAY;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 게임 오버 메인 텍스트
    ctx.fillStyle = '#ff4444';
    ctx.font = 'bold 38px Courier New';
    ctx.textAlign = 'center';
    ctx.shadowBlur = 15;
    ctx.shadowColor = '#ff0000';
    ctx.fillText('GAME OVER', canvas.width / 2, canvas.height / 2 - 40);
    ctx.shadowBlur = 0;

    // 구분선
    ctx.strokeStyle = 'rgba(255, 68, 68, 0.4)';
    ctx.lineWidth = 1;
    ctx.beginPath();
    ctx.moveTo(canvas.width / 2 - 100, canvas.height / 2 - 15);
    ctx.lineTo(canvas.width / 2 + 100, canvas.height / 2 - 15);
    ctx.stroke();

    // 최종 점수 표시
    ctx.fillStyle = '#ffffff';
    ctx.font = '20px Courier New';
    ctx.fillText('최종 점수: ' + score, canvas.width / 2, canvas.height / 2 + 15);

    // 신기록 달성 시 표시
    if (score >= bestScore && score > 0) {
        ctx.fillStyle = '#ffcc00';
        ctx.font = 'bold 14px Courier New';
        ctx.fillText('★ 신기록! ★', canvas.width / 2, canvas.height / 2 + 40);
    }

    // 재시작 안내 메시지
    ctx.fillStyle = '#00ff88';
    ctx.font = '15px Courier New';
    ctx.fillText('[ R ] 재시작', canvas.width / 2, canvas.height / 2 + 68);
    ctx.textAlign = 'left';
}

// 그리드 배경 그리기
function drawGrid() {
    ctx.strokeStyle = COLOR_GRID;
    ctx.lineWidth = 0.5;

    // 세로 그리드 선
    for (let x = 0; x <= GRID_SIZE; x++) {
        ctx.beginPath();
        ctx.moveTo(x * CELL_SIZE, 0);
        ctx.lineTo(x * CELL_SIZE, canvas.height);
        ctx.stroke();
    }

    // 가로 그리드 선
    for (let y = 0; y <= GRID_SIZE; y++) {
        ctx.beginPath();
        ctx.moveTo(0, y * CELL_SIZE);
        ctx.lineTo(canvas.width, y * CELL_SIZE);
        ctx.stroke();
    }
}

// 둥근 사각형을 그리는 헬퍼 함수
function roundRect(x, y, w, h, r) {
    // 모서리 반지름이 너무 크지 않도록 제한
    r = Math.min(r, w / 2, h / 2);
    ctx.beginPath();
    ctx.moveTo(x + r, y);
    ctx.lineTo(x + w - r, y);
    ctx.arcTo(x + w, y,     x + w, y + r,     r); // 우상단 모서리
    ctx.lineTo(x + w, y + h - r);
    ctx.arcTo(x + w, y + h, x + w - r, y + h, r); // 우하단 모서리
    ctx.lineTo(x + r, y + h);
    ctx.arcTo(x,     y + h, x,     y + h - r, r); // 좌하단 모서리
    ctx.lineTo(x, y + r);
    ctx.arcTo(x,     y,     x + r, y,         r); // 좌상단 모서리
    ctx.closePath();
}

// 펄스 효과가 있는 먹이 그리기
function drawFood() {
    // 시간에 따른 펄스 크기 계산 (sin 함수로 주기적 변화)
    const pulse = Math.sin(animFrame * 0.3) * 2;
    const cx = food.x * CELL_SIZE + CELL_SIZE / 2; // 먹이 중심 X
    const cy = food.y * CELL_SIZE + CELL_SIZE / 2; // 먹이 중심 Y
    const radius = CELL_SIZE / 2 - 2 + pulse * 0.5; // 펄스 적용 반지름

    // 외부 발광 효과
    ctx.shadowBlur = 16 + pulse;
    ctx.shadowColor = COLOR_FOOD;

    // 방사형 그라디언트로 먹이 그리기
    const grad = ctx.createRadialGradient(cx - 2, cy - 2, 1, cx, cy, radius);
    grad.addColorStop(0, COLOR_FOOD_INNER); // 중심: 노랑
    grad.addColorStop(1, COLOR_FOOD);       // 외곽: 주황
    ctx.fillStyle = grad;

    // 원형 먹이 그리기
    ctx.beginPath();
    ctx.arc(cx, cy, radius, 0, Math.PI * 2);
    ctx.fill();

    // 반짝이는 하이라이트
    ctx.shadowBlur = 0;
    ctx.fillStyle = 'rgba(255, 255, 255, 0.5)';
    ctx.beginPath();
    ctx.arc(cx - 2, cy - 2, radius * 0.28, 0, Math.PI * 2);
    ctx.fill();
}

// 둥근 세그먼트와 그라디언트 색상으로 뱀 그리기
function drawSnake() {
    const len = snake.length;

    snake.forEach((seg, index) => {
        // 뱀 색상: 머리에서 꼬리로 갈수록 어두워지는 그라디언트
        const t = index / Math.max(len - 1, 1); // 0(머리) ~ 1(꼬리)

        // 머리는 밝은 네온 초록, 꼬리로 갈수록 어두운 초록
        if (index === 0) {
            ctx.fillStyle = COLOR_SNAKE_HEAD; // 머리: 네온 초록
            ctx.shadowBlur = 10;
            ctx.shadowColor = COLOR_SNAKE_HEAD;
        } else if (t < 0.5) {
            ctx.fillStyle = COLOR_SNAKE_MID;  // 앞부분 몸통: 중간 초록
            ctx.shadowBlur = 4;
            ctx.shadowColor = COLOR_SNAKE_MID;
        } else {
            ctx.fillStyle = COLOR_SNAKE_TAIL; // 뒷부분 꼬리: 어두운 초록
            ctx.shadowBlur = 0;
        }

        // 세그먼트 위치 계산 (패딩 1px)
        const x = seg.x * CELL_SIZE + 1;
        const y = seg.y * CELL_SIZE + 1;
        const size = CELL_SIZE - 2;

        // 둥근 사각형으로 세그먼트 그리기
        roundRect(x, y, size, size, CORNER_RADIUS);
        ctx.fill();

        // 그림자 초기화
        ctx.shadowBlur = 0;

        // 머리에만 눈 그리기
        if (index === 0) {
            // 눈 색상: 어두운 배경색
            ctx.fillStyle = '#0a0a14';
            const cx = seg.x * CELL_SIZE + CELL_SIZE / 2;
            const cy = seg.y * CELL_SIZE + CELL_SIZE / 2;
            const eyeOff = 5; // 눈 간격

            // 왼쪽 눈 위치 및 그리기
            ctx.beginPath();
            ctx.arc(
                cx + direction.y * eyeOff - direction.x * 3,
                cy - direction.x * eyeOff - direction.y * 3,
                2.5, 0, Math.PI * 2
            );
            ctx.fill();

            // 오른쪽 눈 위치 및 그리기
            ctx.beginPath();
            ctx.arc(
                cx - direction.y * eyeOff - direction.x * 3,
                cy + direction.x * eyeOff - direction.y * 3,
                2.5, 0, Math.PI * 2
            );
            ctx.fill();
        }
    });
}

// 전체 게임 화면 렌더링
function draw() {
    // 배경 지우기
    ctx.fillStyle = COLOR_BG;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 그리드 그리기
    drawGrid();

    // 먹이 그리기
    drawFood();

    // 뱀 그리기
    drawSnake();
}

// 시작 대기 화면 그리기
function drawStartScreen() {
    // 배경 초기화
    ctx.fillStyle = COLOR_BG;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 그리드 표시
    drawGrid();

    // 네온 제목
    ctx.fillStyle = '#00ff88';
    ctx.shadowBlur = 20;
    ctx.shadowColor = '#00ff88';
    ctx.font = 'bold 44px Courier New';
    ctx.textAlign = 'center';
    ctx.fillText('SNAKE', canvas.width / 2, canvas.height / 2 - 50);
    ctx.shadowBlur = 0;

    // 버전 표시
    ctx.fillStyle = '#00cc66';
    ctx.font = '14px Courier New';
    ctx.fillText('v2 · 개선 버전', canvas.width / 2, canvas.height / 2 - 20);

    // 구분선
    ctx.strokeStyle = 'rgba(0, 255, 136, 0.2)';
    ctx.lineWidth = 1;
    ctx.beginPath();
    ctx.moveTo(canvas.width / 2 - 80, canvas.height / 2 + 5);
    ctx.lineTo(canvas.width / 2 + 80, canvas.height / 2 + 5);
    ctx.stroke();

    // 시작 안내
    ctx.fillStyle = '#9ca3af';
    ctx.font = '17px Courier New';
    ctx.fillText('방향키로 시작', canvas.width / 2, canvas.height / 2 + 32);
    ctx.textAlign = 'left';
}

// 키보드 입력 처리 이벤트 리스너
document.addEventListener('keydown', (e) => {
    // R 키: 재시작 (언제든 가능)
    if (e.key === 'r' || e.key === 'R') {
        initGame();
        return;
    }

    // 게임 미시작 상태에서 방향키 입력 시 게임 시작
    if (!gameStarted) {
        initGame();
    }

    // 방향키 처리 (역방향 이동 금지)
    switch (e.key) {
        case 'ArrowUp':
            // 아래로 이동 중이면 위로 전환 불가
            if (direction.y !== 1) nextDirection = { x: 0, y: -1 };
            e.preventDefault(); // 브라우저 스크롤 막기
            break;
        case 'ArrowDown':
            // 위로 이동 중이면 아래로 전환 불가
            if (direction.y !== -1) nextDirection = { x: 0, y: 1 };
            e.preventDefault();
            break;
        case 'ArrowLeft':
            // 오른쪽으로 이동 중이면 왼쪽 전환 불가
            if (direction.x !== 1) nextDirection = { x: -1, y: 0 };
            e.preventDefault();
            break;
        case 'ArrowRight':
            // 왼쪽으로 이동 중이면 오른쪽 전환 불가
            if (direction.x !== -1) nextDirection = { x: 1, y: 0 };
            e.preventDefault();
            break;
    }
});

// 앱 초기화: 최고 점수 로드 후 시작 화면 표시
loadBestScore();    // localStorage에서 최고 점수 불러오기
drawStartScreen();  // 시작 대기 화면 렌더링
"""

## 🚀 게임 실행

In [ ]:
from IPython.display import IFrame

# CSS, HTML, JS를 하나의 완전한 HTML 파일로 결합
full_html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Snake Game v2</title>
    <style>{css}</style>
</head>
<body>
{html_body}
<script>{js}</script>
</body>
</html>"""

# HTML 파일로 저장
with open('snake_game_v2.html', 'w', encoding='utf-8') as f:
    f.write(full_html)

# IFrame으로 게임 표시
IFrame(src='snake_game_v2.html', width=600, height=680)